## SQL TASK NOTEBOOK 

In [132]:
import sqlite3

db_path = r"C:\Users\LENOVO\OneDrive\Masaüstü\Çalışmalar\Lotus AI\SQL Task\sqlite-sakila.db"  

conn = sqlite3.connect(db_path)

cursor = conn.cursor()

print("Veritabanına başarıyla bağlanıldı.")

Veritabanına başarıyla bağlanıldı.


### Create Dataframes

In [133]:
import pandas as pd

customer = "SELECT * FROM customer"
customer_df = pd.read_sql_query(customer, conn)

category = "SELECT * FROM category"
category_df = pd.read_sql_query(category, conn)

film_category="SELECT * FROM film_category"
film_category_df = pd.read_sql_query(film_category, conn)

language="SELECT * FROM language"
language_df = pd.read_sql_query(language, conn)

actor="SELECT * FROM actor"
actor_df = pd.read_sql_query(actor, conn)   

film_actor="SELECT * FROM film_actor"  
film_actor_df = pd.read_sql_query(film_actor, conn)

film="SELECT * FROM film"
film_df = pd.read_sql_query(film, conn)

staff="SELECT * FROM staff"
staff_df = pd.read_sql_query(staff, conn)

inventory="SELECT * FROM inventory"
inventory_df = pd.read_sql_query(inventory, conn)

payment="SELECT * FROM payment"
payment_df = pd.read_sql_query(payment, conn)

rental="SELECT * FROM rental"
rental_df = pd.read_sql_query(rental, conn)

country="SELECT * FROM country"
country_df=pd.read_sql_query(country,conn)

city="SELECT * FROM city"
city_df=pd.read_sql_query(city,conn)

address="SELECT * FROM address"
address_df=pd.read_sql_query(address,conn)  

store="SELECT * FROM store"
store_df=pd.read_sql_query(store,conn)

print("Tüm tablolar başarıyla DataFrame'e çevirildi.")


Tüm tablolar başarıyla DataFrame'e çevirildi.


## Dataframes:

### customer_df, category_df, film_category_df, language_df, actor_df, film_actor_df, film_df, staff_df, inventory_df, payment_df, rental_df, country_df, city_df, address_df, store_df


## SORULAR

1-) Tüm verileri göster

In [134]:
print(rental_df)

       rental_id              rental_date  inventory_id  customer_id  \
0              1  2005-05-24 22:53:30.000           367          130   
1              2  2005-05-24 22:54:33.000          1525          459   
2              3  2005-05-24 23:03:39.000          1711          408   
3              4  2005-05-24 23:04:41.000          2452          333   
4              5  2005-05-24 23:05:21.000          2079          222   
...          ...                      ...           ...          ...   
16039      16045  2005-08-23 22:25:26.000           772           14   
16040      16046  2005-08-23 22:26:47.000          4364           74   
16041      16047  2005-08-23 22:42:48.000          2088          114   
16042      16048  2005-08-23 22:43:07.000          2019          103   
16043      16049  2005-08-23 22:50:12.000          2666          393   

                   return_date  staff_id          last_update  
0      2005-05-26 22:04:30.000         1  2021-03-06 15:53:41  
1      

2-) Her filmdeki oyuncuları listele

In [135]:
print(actor_df.merge(film_actor_df, on="actor_id")
      .merge(film_df, on="film_id")[["first_name", "last_name", "title"]])

     first_name last_name                  title
0      PENELOPE   GUINESS       ACADEMY DINOSAUR
1      PENELOPE   GUINESS   ANACONDA CONFESSIONS
2      PENELOPE   GUINESS            ANGELS LIFE
3      PENELOPE   GUINESS  BULWORTH COMMANDMENTS
4      PENELOPE   GUINESS          CHEAPER CLYDE
...         ...       ...                    ...
5457      THORA    TEMPLE       TELEGRAPH VOYAGE
5458      THORA    TEMPLE        TROJAN TOMORROW
5459      THORA    TEMPLE        VIRGINIAN PLUTO
5460      THORA    TEMPLE       WARDROBE PHANTOM
5461      THORA    TEMPLE         WRONG BEHAVIOR

[5462 rows x 3 columns]


3-) Her filmde kaç oyuncu oynadı?

In [136]:
result=(film_actor_df.merge(film_df, on="film_id")
    .groupby("title")
    .agg(actor_count=("actor_id", "nunique")).reset_index()
    .sort_values(by="actor_count", ascending=False))
print(result.reset_index(drop=True))

                 title  actor_count
0     LAMBS CINCINATTI           15
1           CRAZY HOME           13
2            RANDOM GO           13
3    BOONDOCK BALLROOM           13
4          CHITTY LOCK           13
..                 ...          ...
992        SHOCK CABIN            1
993    MUSKETEERS WAIT            1
994  FOREVER CANDIDATE            1
995     PACKER MADIGAN            1
996       DWARFS ALTER            1

[997 rows x 2 columns]


Bu soruda dataframe indexlerinin sırasının SQL çıktısıyla aynı index sırasında olması için reset_index() kullandım.

4-) Her oyuncu kaç filmde oynadı?

In [137]:
result=(actor_df.merge(film_actor_df, on="actor_id")
        .groupby(["first_name","last_name","actor_id"])
        .agg(film_count=("film_id", "nunique")).reset_index()
        .sort_values(by="film_count", ascending=False))
print(result.reset_index(drop=True))

    first_name  last_name  actor_id  film_count
0         GINA  DEGENERES       107          42
1       WALTER       TORN       102          41
2         MARY     KEITEL       198          40
3      MATTHEW     CARREY       181          39
4       SANDRA     KILMER        23          37
..         ...        ...       ...         ...
195      SISSY   SOBIESKI        31          18
196      JULIA  ZELLWEGER       186          16
197       JUDY       DEAN        35          15
198      JULIA    FAWCETT       199          15
199      EMILY        DEE       148          14

[200 rows x 4 columns]


Burada ilk deneyişimde Pandas çıktısında 1 tane daha az satır bulunuyordu. Daha sonra fark ettim ki group by yaparken actor_id yi almamışım. Dolayısıyla anlıyoruz ki aynı isim ve soy isime sahip 2 kişi var. Ardından actor_id yi ekleyince hata ortadan kalktı.

5-) Envanterde olmayan filmler var mı? Varsa kaç tane?

In [138]:
result = pd.DataFrame({
    "film_count": [film_df.loc[~film_df["film_id"].isin(inventory_df["film_id"]), "film_id"].nunique()]
})

print(result)

   film_count
0          42


6-) Kiralanabilir olan her filmin kaç kez kiralandığını ve toplam gelirlerini getirin

In [139]:
result = (
    inventory_df.drop(columns=["last_update"])
    .merge(film_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
    .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
    .groupby("title")
    .agg(rental_count=("rental_id", "nunique"), total_revenue=("amount", "sum"))
    .reset_index()
    .sort_values(by="rental_count", ascending=False)
)

print(result.reset_index(drop=True))


                   title  rental_count  total_revenue
0     BUCKET BROTHERHOOD            34         180.66
1       ROCKETEER MOTHER            33         116.67
2    RIDGEMONT SUBMARINE            32         130.68
3         GRIT CLOCKWORK            32         110.68
4         JUGGLER HARDLY            32          96.68
..                   ...           ...            ...
953      FULL FLATLINERS             5          16.95
954     BRAVEHEART HUMAN             5          16.95
955       HARDLY ROBBERS             4          15.96
956          MIXED DOORS             4          15.96
957          TRAIN BUNCH             4          24.96

[958 rows x 3 columns]


Burada önce hiç drop kullanmadan yapmaya çalıştım ama her tabloda last_update kolonu bulunduğu için ve 4 tablo kullanıldığından dolayı conflictler yaşandı. dolayısıyla ben de last update kolonunu tamamıyla kaldırdım. 

7-) Envanterde olmayan filmlerin kira oranlarını getirin

In [140]:
result=film_df[film_df["film_id"].isin(inventory_df["film_id"]) == False][["title", "rental_rate"]]
print(result.reset_index(drop=True))

                     title  rental_rate
0           ALICE FANTASIA         0.99
1              APOLLO TEEN         2.99
2           ARGONAUTS TOWN         0.99
3            ARK RIDGEMONT         0.99
4     ARSENIC INDEPENDENCE         0.99
5        BOONDOCK BALLROOM         0.99
6            BUTCH PANTHER         0.99
7            CATCH AMISTAD         0.99
8      CHINATOWN GLADIATOR         4.99
9           CHOCOLATE DUCK         2.99
10    COMMANDMENTS EXPRESS         4.99
11        CROSSING DIVORCE         4.99
12         CROWDS TELEMARK         4.99
13        CRYSTAL BREAKING         2.99
14              DAZED PUNK         4.99
15  DELIVERANCE MULHOLLAND         0.99
16       FIREHOUSE VIETNAM         0.99
17           FLOATS GARDEN         2.99
18   FRANKENSTEIN STRANGER         0.99
19      GLADIATOR WESTWARD         4.99
20               GUMP DATE         4.99
21           HATE HANDICAP         0.99
22             HOCUS FRIDA         2.99
23        KENTUCKIAN GIANT         2.99


8-) Birden fazla DVD'yi iade etmeyen kaç müşteri var?

In [141]:
result=(rental_df[rental_df["return_date"].isnull()].groupby("customer_id")
.agg(active_rentals=("rental_id","count"))
.query("active_rentals>1"))
print(len(result.reset_index(drop=True)))

23


Burada len ile farklı değerleri belirttiğim için kolon ismini yazdıramadım

9-) Her müşteri kaç film kiraladı?

In [142]:
result=(customer_df.merge(rental_df, on="customer_id")
        .groupby(["customer_id","first_name","last_name"])
        .agg(rental_count=("rental_id", "count")).reset_index()
        .sort_values(by="rental_count", ascending=False))
print(result.reset_index(drop=True))

     customer_id first_name last_name  rental_count
0            148    ELEANOR      HUNT            46
1            526       KARL      SEAL            45
2            144      CLARA      SHAW            42
3            236     MARCIA      DEAN            42
4             75      TAMMY   SANDERS            41
..           ...        ...       ...           ...
594          248   CAROLINE    BOWMAN            15
595          110    TIFFANY    JORDAN            14
596           61  KATHERINE    RIVERA            14
597          281      LEONA    OBRIEN            14
598          318      BRIAN     WYMAN            12

[599 rows x 4 columns]


10-) Türlerine göre en çok kiralanan filmler ve bunlara ne kadar ödendi?

In [143]:
result = (
    film_df
    .merge(film_category_df.drop(columns=["last_update"]), on="film_id")
    .merge(category_df.drop(columns=["last_update"]), on="category_id")
    .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
    .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
    .rename(columns={"name": "genre"})
    .groupby(["genre", "title"])
    .agg(
        rental_count=("rental_id", "count"),
        total_revenue=("amount", "sum")
    )
    .reset_index()
    .sort_values(["rental_count", "title"], ascending=[False, True])    .reset_index(drop=True)
)

print(result)

           genre               title  rental_count  total_revenue
0         Travel  BUCKET BROTHERHOOD            34         180.66
1        Foreign    ROCKETEER MOTHER            33         116.67
2          Games      FORWARD TEMPLE            32         128.68
3          Games      GRIT CLOCKWORK            32         110.68
4      Animation      JUGGLER HARDLY            32          96.68
..           ...                 ...           ...            ...
953        Games         SEVEN SWARM             5          30.95
954       Travel      TRAFFIC HOBBIT             5          24.95
955  Documentary      HARDLY ROBBERS             4          15.96
956      Foreign         MIXED DOORS             4          15.96
957       Horror         TRAIN BUNCH             4          24.96

[958 rows x 4 columns]


Bu şu ana kadar üzerine en çok uğraştığım problem oldu. Her bir kategori içerisindkei en fazla satış yapılan filmi bulmam gerektiğini düşünüp üzerine çok zaman harcadım ancak yanlış ilerlemişim. Daha sonrasında olması gereken haline getirdim.

11-) Tür ve tarihe göre kiralama sayısı ve gelir

In [144]:
result = (
    film_df
    .merge(film_category_df.drop(columns=["last_update"]), on="film_id")
    .merge(category_df.drop(columns=["last_update"]), on="category_id")
    .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
    .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
    .rename(columns={"name": "genre"})
    .assign(rental_date=lambda df: pd.to_datetime(df["rental_date"]).dt.date)
    .groupby(["genre", "rental_date"])
    .agg(
        rental_count=("rental_id", "count"),
        total_revenue=("amount", "sum")
    )
    .reset_index()
    .sort_values("rental_date", ascending=False)
    .reset_index(drop=True)
)

print(result)

           genre rental_date  rental_count  total_revenue
0         Travel  2006-02-14            10          30.91
1    Documentary  2006-02-14             6           7.96
2       Classics  2006-02-14             9          24.89
3         Family  2006-02-14            13          29.90
4        Foreign  2006-02-14            11          24.89
..           ...         ...           ...            ...
628        Music  2005-05-24             1           2.99
629     Children  2005-05-24             2          10.98
630       Horror  2005-05-24             2           6.98
631       Family  2005-05-24             1           2.99
632    Animation  2005-05-24             1           4.99

[633 rows x 4 columns]


Burada SQL sorgularıyla arasında farklılık var. Veri boyutu tamamen aynı ve her ikisi de DESC sıralanmasına rağmen sıralamalar farklı. Bunun sebebi de date i sadece gün-ay-yıl olacak sekilde kırpmış olmam çünkü kırpmadan önce Pandas işlemlerinde 600 SQL işlemlerinde neredeyse 16000 cıktı vardı (date formatından kaynaklı). Daha sonrasında iki tarafı da aynı formata getirdim ve çıktılar eşleşti.

12-) Kiralanabilir filmler için türlerine göre her filmin kaç kez kiralandığı

In [145]:
result=(film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
        .merge(category_df.drop(columns=["last_update"]), on="category_id")
        .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
        .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
        .groupby(["name", "title"])
        .size()
        .reset_index(name="rental_count"))
print(result.reset_index(drop=True))

       name                title  rental_count
0    Action         AMADEUS HOLY            21
1    Action      AMERICAN CIRCUS            22
2    Action   ANTITRUST TOMATOES            10
3    Action  BAREFOOT MANCHURIAN            18
4    Action         BERETS AGENT            21
..      ...                  ...           ...
953  Travel  VALENTINE VANISHING            12
954  Travel          WINDOW SIDE            12
955  Travel        WOLVES DESIRE            21
956  Travel        WORKER TARZAN            15
957  Travel  WORKING MICROCOSMOS            25

[958 rows x 3 columns]


13-) En çok rafta bekleyen filmler

In [146]:
result = (
    film_df
    .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
)

result = result[result["return_date"].isnull()]

result["rental_date"] = pd.to_datetime(result["rental_date"])
result["days_on_shelf"] = (
    pd.Timestamp.now() - result["rental_date"]
).dt.days

result = result[["title", "days_on_shelf"]]

result = result.sort_values(by="days_on_shelf", ascending=False)

print(result.reset_index(drop=True))

                     title  days_on_shelf
0         ACADEMY DINOSAUR           7624
1          MUMMY CREATURES           7446
2             NONE SPIKING           7446
3      OPERATION OPERATION           7446
4        OPPOSITE NECKLACE           7446
..                     ...            ...
178           FAMILY SWEET           7446
179       FIGHT JAWBREAKER           7446
180            FLYING HOOK           7446
181  FORRESTER COMANCHEROS           7446
182           ZHIVAGO CORE           7446

[183 rows x 2 columns]


Pandasta daha önce date işlemi yapmadığım için burada AI dan yardım aldım.

14-) Geç, erken ve zamanında iade edilmiş kiralanmış filmler

In [147]:
rental_df["rental_date"] = pd.to_datetime(rental_df["rental_date"])
rental_df["return_date"] = pd.to_datetime(rental_df["return_date"])

merged = (rental_df.merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .dropna(subset=["return_date"]))

merged["due_date"] = (merged["rental_date"] + pd.to_timedelta(merged["rental_duration"], unit="D")).dt.normalize()
merged["return_date_norm"] = merged["return_date"].dt.normalize()

result = pd.DataFrame({
    "late_returns": [(merged["return_date_norm"] > merged["due_date"]).sum()],
    "early_returns": [(merged["return_date_norm"] < merged["due_date"]).sum()],
    "on_time_returns": [(merged["return_date_norm"] == merged["due_date"]).sum()]
})
print(result)

   late_returns  early_returns  on_time_returns
0          7269           6864             1728


Burada veritabanında doğrudan bir bilgi olmadığı için hesaplamamız gerekiyordu. Pandasta bu işlemi yapmak SQL den daha zorladı. Çıktılar ilk farklıydı ancak yine zaman formatlarını aynı yapınca eşitlendiler.

15-) Hangi müşteri en çok DVD kiralamış?

In [148]:
result=rental_df.merge(customer_df, on="customer_id").groupby(["customer_id","first_name","last_name"]).agg(total_rentals=("rental_id","count")).reset_index().sort_values(by="total_rentals", ascending=False)
print(result.reset_index(drop=True).head(1))

   customer_id first_name last_name  total_rentals
0          148    ELEANOR      HUNT             46


16-)En popüler film kategorisi nedir?

In [149]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .groupby("name")
          .size()
          .reset_index(name="rental_count")
          .sort_values("rental_count", ascending=False)
          .head(1))
print(result.reset_index(drop=True))

     name  rental_count
0  Sports          1179


17-) Hangi çalışan en çok kiralama işlemi gerçekleştirmiş?

In [150]:
result=(
    rental_df.merge(staff_df.drop(columns=["last_update"]), on="staff_id")
    .groupby(["first_name", "last_name"])
    .size()
    .reset_index(name="total_rental")
    .sort_values("total_rental", ascending=False)
    .head(1)
)
print(result.reset_index(drop=True))

  first_name last_name  total_rental
0       Mike   Hillyer          8040


18-) En çok geliri hangi film getirmiş?

In [151]:
result=(
    film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
    .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
    .groupby("title")
    .agg(total_revenue=("amount", "sum"))
    .reset_index()
    .sort_values("total_revenue", ascending=False)
    .head(1)
)
print(result.reset_index(drop=True))

              title  total_revenue
0  TELEGRAPH VOYAGE         231.73


19-) Her müşteri için toplam harcama miktarı

In [152]:
reesult=(
    customer_df.merge(payment_df.drop(columns=["last_update"]), on="customer_id")
    .groupby(["first_name", "last_name"])
    .agg(total_spent=("amount", "sum"))
    .reset_index()
    .sort_values("total_spent", ascending=False)
)
print(reesult.reset_index(drop=True))

    first_name last_name  total_spent
0         KARL      SEAL       221.55
1      ELEANOR      HUNT       216.54
2        CLARA      SHAW       195.58
3       RHONDA   KENNEDY       194.61
4       MARION    SNYDER       194.61
..         ...       ...          ...
594      ANNIE   RUSSELL        58.82
595     JOHNNY    TURPIN        57.81
596      BRIAN     WYMAN        52.88
597      LEONA    OBRIEN        50.86
598   CAROLINE    BOWMAN        50.85

[599 rows x 3 columns]


20-) Her kategorideki toplam kiralama sayısını ve gelirleri bululn

In [153]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
          .groupby("name")
          .agg(rental_count=("rental_id", "count"), total_revenue=("amount", "sum"))
          .sort_values("total_revenue",ascending=False)
          .reset_index())
print(result)

           name  rental_count  total_revenue
0        Sports          1179        5314.21
1        Sci-Fi          1101        4756.98
2     Animation          1166        4656.30
3         Drama          1060        4587.39
4        Comedy           941        4383.58
5        Action          1112        4375.85
6           New           940        4351.62
7         Games           969        4281.33
8       Foreign          1033        4270.67
9        Family          1096        4226.07
10  Documentary          1050        4217.52
11       Horror           846        3722.54
12     Children           945        3655.55
13     Classics           939        3639.59
14       Travel           837        3549.64
15        Music           830        3417.72


21-) En uzun süre kirada kalmış filmleri bulun

In [154]:
rental_df["rental_date"] = pd.to_datetime(rental_df["rental_date"])
rental_df["return_date"] = pd.to_datetime(rental_df["return_date"])

result = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .dropna(subset=["return_date"]))

result["duration"] = (result["return_date"] - result["rental_date"]) / pd.Timedelta(days=1)

result = (result[["title", "duration"]]
          .sort_values("duration", ascending=False))
print(result.reset_index(drop=True))

                    title  duration
0      HOLOCAUST HIGHBALL  9.249306
1         HIGHBALL POTTER  9.249306
2              MASK PEACH  9.248611
3            TRAMP OTHERS  9.248611
4              PANIC CLUB  9.248611
...                   ...       ...
15856      STRAIGHT HOURS  0.751389
15857           RACER EGG  0.751389
15858        ROAD ROXANNE  0.750694
15859     DISCIPLE MOTHER  0.750000
15860   CURTAIN VIDEOTAPE  0.750000

[15861 rows x 2 columns]


22-) En az kiralanan 5 film hangileridir?

In [155]:
result=(
    film_df.merge(inventory_df.drop(columns=["last_update"]),on="film_id")
    .merge(rental_df.drop(columns=["last_update"]),on="inventory_id")
    .groupby("title")
    .agg(rental_count=("rental_id","count"))
    .sort_values(["rental_count","title"],ascending=[True,True])
    .reset_index()

)
print(result.head(5))

              title  rental_count
0    HARDLY ROBBERS             4
1       MIXED DOORS             4
2       TRAIN BUNCH             4
3  BRAVEHEART HUMAN             5
4       BUNCH MINDS             5


24-) En fazla kazanç sağlayan 5 müşteriyi bulun

In [156]:
result=(
    customer_df.merge(payment_df.drop(columns=["last_update"]),on="customer_id")
    .groupby(["customer_id","first_name","last_name"])
    .agg(total=("amount","sum"))
    .sort_values("total",ascending=False)
    .reset_index()
         
)
print(result.head(5))

   customer_id first_name last_name   total
0          526       KARL      SEAL  221.55
1          148    ELEANOR      HUNT  216.54
2          144      CLARA      SHAW  195.58
3          137     RHONDA   KENNEDY  194.61
4          178     MARION    SNYDER  194.61


25-) Her filmin ortalama kiralanma süresini bulun

In [157]:
rental_df["rental_date"] = pd.to_datetime(rental_df["rental_date"])
rental_df["return_date"] = pd.to_datetime(rental_df["return_date"])

result = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .dropna(subset=["return_date"]))

result["duration"] = (result["return_date"] - result["rental_date"]) / pd.Timedelta(days=1)

result = (result.groupby("title")
          .agg(avg_rental_duration=("duration", "mean"))
          .reset_index())
print(result)

                 title  avg_rental_duration
0     ACADEMY DINOSAUR             4.997128
1       ACE GOLDFINGER             5.642708
2     ADAPTATION HOLES             3.465625
3     AFFAIR PREJUDICE             4.758333
4          AFRICAN EGG             7.106629
..                 ...                  ...
953     YOUNG LANGUAGE             4.624405
954         YOUTH KICK             5.562963
955       ZHIVAGO CORE             5.950087
956  ZOOLANDER FICTION             5.542647
957          ZORRO ARK             4.539180

[958 rows x 2 columns]


26-) Her türde en popüler filmi bulun

In [158]:
result = (
    film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
    .merge(category_df.drop(columns=["last_update"]), on="category_id")
    .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
    .rename(columns={"name": "genre"})
    .groupby("genre", as_index=False, sort=False)          # sort=False -> SQL'deki gibi sort etme
    .agg(title=("title", "first"), count=("rental_id", "count"))
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
print(result)

          genre                title  count
0        Sports     ALADDIN CALENDAR   1179
1     Animation        ALTER VICTORY   1166
2        Action         AMADEUS HOLY   1112
3        Sci-Fi       ANNIE IDENTITY   1101
4        Family          AFRICAN EGG   1096
5         Drama        BEAUTY GREASE   1060
6   Documentary     ACADEMY DINOSAUR   1050
7       Foreign         AGENT TRUMAN   1033
8         Games          AUTUMN CROW    969
9      Children  BACKLASH UNDEFEATED    945
10       Comedy      AIRPLANE SIERRA    941
11          New    AMISTAD MIDSUMMER    940
12     Classics         ARIZONA BANG    939
13       Horror       ACE GOLDFINGER    846
14       Travel           BASIC EASY    837
15        Music       ALASKA PHANTOM    830


Genre ve countlar doğru ancak  title lar sql den farklı. Sebebini araştırdığımda SQLite ın aynı sayıda ögeler için seçim işlemini rasgele yaptığı gibi bir şeyle karşılaştım. Dolayısıyla bu farklılığın buradan geldiğini düşünüyorum

27-) Her türde en fazla gelir sağlayan filmi bulun

In [232]:
result = (
    film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
    .merge(category_df.drop(columns=["last_update"]), on="category_id")
    .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
    .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
    .rename(columns={"name": "genre"})
    .groupby("genre", as_index=False, sort=False)
    .agg(title=("title", "first"), revenue=("amount", "sum"))
    .sort_values("revenue", ascending=False)
    .reset_index(drop=True)
)
print(result)

          genre                title  revenue
0        Sports     ALADDIN CALENDAR  5314.21
1        Sci-Fi       ANNIE IDENTITY  4756.98
2     Animation        ALTER VICTORY  4656.30
3         Drama        BEAUTY GREASE  4587.39
4        Comedy      AIRPLANE SIERRA  4383.58
5        Action         AMADEUS HOLY  4375.85
6           New    AMISTAD MIDSUMMER  4351.62
7         Games          AUTUMN CROW  4281.33
8       Foreign         AGENT TRUMAN  4270.67
9        Family          AFRICAN EGG  4226.07
10  Documentary     ACADEMY DINOSAUR  4217.52
11       Horror       ACE GOLDFINGER  3722.54
12     Children  BACKLASH UNDEFEATED  3655.55
13     Classics         ARIZONA BANG  3639.59
14       Travel           BASIC EASY  3549.64
15        Music       ALASKA PHANTOM  3417.72


Yukarıdaki sebeple aynı sebepten dolayı cıktılar farklı

28-) En çok DVD iade etmeyen müşteriyi bulun

In [160]:
unreturned_df = rental_df[rental_df["return_date"].isna()]

result = (customer_df.merge(unreturned_df.drop(columns=["last_update"]), on="customer_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="unreturned_count")
          .sort_values("unreturned_count", ascending=False)
          .reset_index(drop=True)
          .head(1))
print(result)

  first_name last_name  unreturned_count
0      TAMMY   SANDERS                 3


29-) En fazla kiralama yapan 5 çalışanı bulun

In [161]:
result = (staff_df.merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="rental_count")
          .sort_values("rental_count", ascending=False)
          .reset_index(drop=True)
          .head(5))
print(result)

  first_name last_name  rental_count
0       Mike   Hillyer          8040
1        Jon  Stephens          8004


30-) En fazla kiralama yapan 5 müşteri hangi şubeden kiralama yapmıştır

In [162]:
result = (customer_df.drop(columns=["store_id"])
          .merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(store_df.drop(columns=["last_update"]), on="store_id")
          .groupby(["store_id", "first_name", "last_name"])
          .size()
          .reset_index(name="rental_count")
          .sort_values("rental_count", ascending=False)
          .reset_index(drop=True)
          .head(5))
print(result)

   store_id first_name last_name  rental_count
0         2      JORGE  OLIVARES            26
1         1   GERTRUDE  CASTILLO            25
2         1     MARCIA      DEAN            25
3         2      CLARA      SHAW            25
4         2    ELEANOR      HUNT            25


31-) Her türde en az kiralanan filmi bulun

In [233]:
result = (
    category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
    .merge(film_df.drop(columns=["last_update"]), on="film_id")
    .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
    .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
    .rename(columns={"name": "genre"})
    .groupby("genre", as_index=False, sort=False)
    .agg(title=("title", "first"), rental_count=("rental_id", "count"))
    .sort_values("rental_count", ascending=True)
    .reset_index(drop=True)
)
print(result)

          genre                title  rental_count
0         Music       ALASKA PHANTOM           830
1        Travel           BASIC EASY           837
2        Horror       ACE GOLDFINGER           846
3      Classics         ARIZONA BANG           939
4           New    AMISTAD MIDSUMMER           940
5        Comedy      AIRPLANE SIERRA           941
6      Children  BACKLASH UNDEFEATED           945
7         Games          AUTUMN CROW           969
8       Foreign         AGENT TRUMAN          1033
9   Documentary     ACADEMY DINOSAUR          1050
10        Drama        BEAUTY GREASE          1060
11       Family          AFRICAN EGG          1096
12       Sci-Fi       ANNIE IDENTITY          1101
13       Action         AMADEUS HOLY          1112
14    Animation        ALTER VICTORY          1166
15       Sports     ALADDIN CALENDAR          1179


Yine SQLite farklılığından dolayı çıktılar farklıdır.

32-) En çok kiralama yapan 5 müşteri hangi şehirde?

In [164]:
result = (customer_df.merge(address_df.drop(columns=["last_update"]), on="address_id")
          .merge(city_df.drop(columns=["last_update"]), on="city_id")
          .merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .groupby(["city", "first_name", "last_name"])
          .size()
          .reset_index(name="rental_count")
          .sort_values("rental_count", ascending=False)
          .reset_index(drop=True)
          .head(5))
print(result)

          city first_name last_name  rental_count
0  Saint-Denis    ELEANOR      HUNT            46
1   Cape Coral       KARL      SEAL            45
2    Molodetno      CLARA      SHAW            42
3        Tanza     MARCIA      DEAN            42
4     Changhwa      TAMMY   SANDERS            41


33-)En çok kazanç sağlayan 5 müşteri hangi şehirde

In [165]:
result = (customer_df.merge(address_df.drop(columns=["last_update"]), on="address_id")
          .merge(city_df.drop(columns=["last_update"]), on="city_id")
          .merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
          .groupby(["city", "first_name", "last_name"])
          .agg(total_spent=("amount", "sum"))
          .reset_index()
          .sort_values("total_spent", ascending=False)
          .reset_index(drop=True)
          .head(5))
print(result)

                  city first_name last_name  total_spent
0           Cape Coral       KARL      SEAL       221.55
1          Saint-Denis    ELEANOR      HUNT       216.54
2            Molodetno      CLARA      SHAW       195.58
3            Apeldoorn     RHONDA   KENNEDY       194.61
4  Santa Brbara dOeste     MARION    SNYDER       194.61


34-) En çok kiralanan 5 film hangi şehirde

In [166]:
result = (city_df.merge(address_df.drop(columns=["last_update"]), on="city_id")
          .merge(customer_df.drop(columns=["last_update", "store_id"]), on="address_id")
          .merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .groupby(["city", "title"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "title"], ascending=[False, True])
          .reset_index(drop=True)
          .head(5))
print(result)

       city              title  rental_count
0  Sorocaba    CADDYSHACK JEDI             3
1    Kurgan   DETECTIVE VISION             3
2      Lima    DISCIPLE MOTHER             3
3   Trshavn  FLATLINERS KILLER             3
4   Caracas   AFFAIR PREJUDICE             2


35-) En az kiralanan 5 film hangi şehirde

In [167]:
result = (city_df.merge(address_df.drop(columns=["last_update"]), on="city_id")
          .merge(customer_df.drop(columns=["last_update", "store_id"]), on="address_id")
          .merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .groupby(["city", "title"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "title"], ascending=[True, True])
          .head(5)
          .reset_index(drop=True))
print(result)

               city             title  rental_count
0   Almirante Brown  ACADEMY DINOSAUR             1
1            Celaya  ACADEMY DINOSAUR             1
2  Charlotte Amalie  ACADEMY DINOSAUR             1
3           Cianjur  ACADEMY DINOSAUR             1
4            Goinia  ACADEMY DINOSAUR             1


36-) En çok kazanç sağlayan 5 film hangi şehirde

In [168]:
result = (city_df.merge(address_df.drop(columns=["last_update"]), on="city_id")
          .merge(customer_df.drop(columns=["last_update"]), on="address_id")
          .merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
          .groupby(["city", "title"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values("total_revenue", ascending=False)
          .reset_index(drop=True)
          .head(5))
print(result)

                  city             title  total_revenue
0          Probolinggo      MINDS TRUMAN          18.98
1                Plock  CALIFORNIA BIRDS          18.98
2               Kamyin    ROSES TREASURE          17.98
3  Santa Brbara dOeste         WIFE TURN          17.98
4               Datong      EAGLES PANKY          16.98


37-) En az kazanç sağlayan 5 film hangi şehirde

In [169]:
result = (city_df.merge(address_df.drop(columns=["last_update"]), on="city_id")
          .merge(customer_df.drop(columns=["last_update", "store_id"]), on="address_id")
          .merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby(["city", "title"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "title"], ascending=[True, True])
          .head(5)
          .reset_index(drop=True))
print(result)

        city              title  total_revenue
0      Merlo       BLADE POLISH            0.0
1     Jaipur    CHAMBER ITALIAN            0.0
2       Bern      CHASING FIGHT            0.0
3   Rockford    CLEOPATRA DEVIL            0.0
4  Balurghat  CURTAIN VIDEOTAPE            0.0


Total revenuesi 0 olan çok fazla film olduğu için hem SQL de hem de Pandas da 2. bir sort parametresi ekledim ki çıktılar aynı olsunlar.

38-)En fazla kiralama yapan müşteri hangi filmleri kiralamış

In [170]:
top_customer_id = (rental_df.groupby("customer_id")
                    .size()
                    .reset_index(name="rental_count")
                    .sort_values("rental_count", ascending=False)
                    .head(1)["customer_id"].iloc[0])

merged = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .rename(columns={"name": "genre"}))

result = (merged[merged["customer_id"] == top_customer_id]
          .groupby(["first_name", "last_name", "genre"])
          .size()
          .reset_index(name="rental_count"))
print(result)

   first_name last_name        genre  rental_count
0     ELEANOR      HUNT       Action             3
1     ELEANOR      HUNT     Classics             4
2     ELEANOR      HUNT       Comedy             2
3     ELEANOR      HUNT  Documentary             3
4     ELEANOR      HUNT        Drama             1
5     ELEANOR      HUNT       Family             6
6     ELEANOR      HUNT      Foreign             4
7     ELEANOR      HUNT        Games             3
8     ELEANOR      HUNT       Horror             4
9     ELEANOR      HUNT        Music             1
10    ELEANOR      HUNT          New             2
11    ELEANOR      HUNT       Sci-Fi             7
12    ELEANOR      HUNT       Sports             1
13    ELEANOR      HUNT       Travel             5


40-)En çok kazanç sağlayan müşteri hangi filmleri kiralamış

In [171]:
top_customer_id = (rental_df.merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
                    .groupby("customer_id")
                    .agg(total_spent=("amount", "sum"))
                    .reset_index()
                    .sort_values("total_spent", ascending=False)
                    .head(1)["customer_id"].iloc[0])

merged = (customer_df.drop(columns=["store_id"]).merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id"))

result = (merged[merged["customer_id"] == top_customer_id]
          .groupby(["first_name", "last_name", "title"])
          .agg(total_spent=("amount", "sum"))
          .reset_index())
print(result)

   first_name last_name                  title  total_spent
0        KARL      SEAL          BETRAYED REAR         4.99
1        KARL      SEAL       BIKINI BORROWERS         6.99
2        KARL      SEAL         BLUES INSTINCT         2.99
3        KARL      SEAL    BRINGING HYSTERICAL         2.99
4        KARL      SEAL         CYCLONE FAMILY         4.99
5        KARL      SEAL             DATE SPEED         3.99
6        KARL      SEAL           DESIRE ALIEN         2.99
7        KARL      SEAL       DESTINY SATURDAY         4.99
8        KARL      SEAL             DUMBO LUST         0.99
9        KARL      SEAL           DURHAM PANKY         4.99
10       KARL      SEAL       ENGLISH BULWORTH         5.99
11       KARL      SEAL         FIDELITY DEVIL         4.99
12       KARL      SEAL       FOOL MOCKINGBIRD        10.99
13       KARL      SEAL  FORRESTER COMANCHEROS         4.99
14       KARL      SEAL            GANGS PRIDE         2.99
15       KARL      SEAL        GOLDMINE 

41-) En az kazanç sağlayan müşteri hangi filmleri kiralamış

In [172]:
bottom_customer_id = (rental_df
                       .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
                       .groupby("customer_id")
                       .agg(total_spent=("amount", "sum"))
                       .reset_index()
                       .sort_values("total_spent", ascending=True)
                       .head(1)["customer_id"].iloc[0])

merged = (customer_df.drop(columns=["store_id"]).merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id"))

result = (merged[merged["customer_id"] == bottom_customer_id]
          .groupby(["first_name", "last_name", "title"])
          .agg(total_spent=("amount", "sum"))
          .reset_index())
print(result)

   first_name last_name                   title  total_spent
0    CAROLINE    BOWMAN         ARMAGEDDON LOST         0.99
1    CAROLINE    BOWMAN        CAMELOT VACATION         0.99
2    CAROLINE    BOWMAN         DISCIPLE MOTHER         5.99
3    CAROLINE    BOWMAN        EMPIRE MALKOVICH         0.99
4    CAROLINE    BOWMAN          FRENCH HOLIDAY         7.99
5    CAROLINE    BOWMAN       GOODFELLAS SALUTE         4.99
6    CAROLINE    BOWMAN       HOOSIERS BIRDCAGE         4.99
7    CAROLINE    BOWMAN         ILLUSION AMELIE         0.99
8    CAROLINE    BOWMAN    IMPOSSIBLE PREJUDICE         5.99
9    CAROLINE    BOWMAN              JADE BUNCH         2.99
10   CAROLINE    BOWMAN         KILLER INNOCENT         2.99
11   CAROLINE    BOWMAN              MASK PEACH         3.99
12   CAROLINE    BOWMAN  RESURRECTION SILVERADO         0.99
13   CAROLINE    BOWMAN             TEXAS WATCH         0.99
14   CAROLINE    BOWMAN         WHISPERER GIANT         4.99


42-) En az kazanç sağlayan müşteri hangi türde en fazla film kiralamış

In [173]:
bottom_customer_id = (rental_df.merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
                       .groupby("customer_id")
                       .agg(total_spent=("amount", "sum"))
                       .reset_index()
                       .sort_values("total_spent", ascending=True)
                       .head(1)["customer_id"].iloc[0])

merged = (customer_df.drop(columns=["store_id"]).merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .rename(columns={"name": "genre"}))

result = (merged[merged["customer_id"] == bottom_customer_id]
          .groupby(["first_name", "last_name", "genre"])
          .size()
          .reset_index(name="rental_count")
          .sort_values("rental_count", ascending=False)
          .reset_index(drop=True)
          .head(1))
print(result)

  first_name last_name   genre  rental_count
0   CAROLINE    BOWMAN  Sci-Fi             4


43-) En çok kiralanan film hangi çalışan tarafından kiralanmış

In [174]:
top_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
               .groupby("film_id")
               .size()
               .reset_index(name="rental_count")
               .sort_values("rental_count", ascending=False)
               .head(1)["film_id"].iloc[0])

merged = (staff_df.merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id"))

result = (merged[merged["film_id"] == top_film_id]
          .groupby(["first_name", "last_name", "title"])
          .size()
          .reset_index(name="rental_count"))
print(result)

  first_name last_name               title  rental_count
0        Jon  Stephens  BUCKET BROTHERHOOD            18
1       Mike   Hillyer  BUCKET BROTHERHOOD            16


44-) En az kiralanan film hangi çalışan tarafından kiralanmış

In [175]:
bottom_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
                   .groupby("film_id")
                   .size()
                   .reset_index(name="rental_count")
                   .merge(film_df[["film_id", "title"]], on="film_id")      # EKLENDİ
                   .sort_values(["rental_count", "title"], ascending=[True, True])  # DEĞİŞTİ
                   .head(1)["film_id"].iloc[0])

merged = (staff_df.merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id"))

result = (merged[merged["film_id"] == bottom_film_id]
          .groupby(["first_name", "last_name", "title"])
          
          .size()
          .reset_index(name="rental_count"))
print(result)

  first_name last_name           title  rental_count
0        Jon  Stephens  HARDLY ROBBERS             3
1       Mike   Hillyer  HARDLY ROBBERS             1


45-) En çok kazanç sağlayan film hangi çalışan tarafından kiralanmış

In [176]:
top_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
               .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
               .groupby("film_id")
               .agg(total_revenue=("amount", "sum"))
               .reset_index()
               .sort_values("total_revenue", ascending=False)
               .head(1)["film_id"].iloc[0])

merged = (staff_df.merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id"))

result = (merged[merged["film_id"] == top_film_id]
          .groupby(["first_name", "last_name", "title"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index())
print(result)

  first_name last_name             title  total_revenue
0        Jon  Stephens  TELEGRAPH VOYAGE         147.83
1       Mike   Hillyer  TELEGRAPH VOYAGE          83.90


46-)En az kazanç sağlayan film hangi çalışan tarafından kiralanmış


In [177]:
bottom_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
                   .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
                   .groupby("film_id")
                   .agg(total_revenue=("amount", "sum"))
                   .reset_index()
                   .merge(film_df[["film_id", "title"]], on="film_id")      # EKLENDİ
                   .sort_values(["total_revenue", "title"], ascending=[True, True])  # DEĞİŞTİ
                   .head(1)["film_id"].iloc[0])

merged = (staff_df.merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id"))

result = (merged[merged["film_id"] == bottom_film_id]
          .groupby(["first_name", "last_name", "title"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index())
print(result)

  first_name last_name             title  total_revenue
0        Jon  Stephens  OKLAHOMA JUMANJI           1.98
1       Mike   Hillyer  OKLAHOMA JUMANJI           3.96


47-)En çok kiralanan film hangi mağazada kiralanmış

In [178]:
top_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
               .groupby("film_id")
               .size()
               .reset_index(name="rental_count")
               .sort_values("rental_count", ascending=False)
               .head(1)["film_id"].iloc[0])

merged = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id"))

result = (merged[merged["film_id"] == top_film_id]
          .groupby(["store_id", "title"])
          .size()
          .reset_index(name="rental_count"))
print(result)

   store_id               title  rental_count
0         1  BUCKET BROTHERHOOD            16
1         2  BUCKET BROTHERHOOD            18


48-) En az kiralanan film hangi mağazada kiralanmış

In [179]:
bottom_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
                   .groupby("film_id")
                   .size()
                   .reset_index(name="rental_count")
                   .merge(film_df[["film_id", "title"]], on="film_id")      # EKLENDİ
                   .sort_values(["rental_count", "title"], ascending=[True, True])  # DEĞİŞTİ
                   .head(1)["film_id"].iloc[0])

merged = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id"))

result = (merged[merged["film_id"] == bottom_film_id]
          .groupby(["store_id", "title"])
          .size()
          .reset_index(name="rental_count"))
print(result)

   store_id           title  rental_count
0         1  HARDLY ROBBERS             1
1         2  HARDLY ROBBERS             3


49-) En çok kazanç sağlayan film hangi mağazada kiralanmış

In [180]:
top_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
               .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
               .groupby("film_id")
               .agg(total_revenue=("amount", "sum"))
               .reset_index()
               .sort_values("total_revenue", ascending=False)
               .head(1)["film_id"].iloc[0])

merged = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id"))

result = (merged[merged["film_id"] == top_film_id]
          .groupby(["store_id", "title"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index())
print(result)

   store_id             title  total_revenue
0         1  TELEGRAPH VOYAGE          83.90
1         2  TELEGRAPH VOYAGE         147.83


50-) En az kazanç sağlayan film hangi mağazada kiralanmış

In [181]:
bottom_film_id = (inventory_df.merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
                   .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
                   .groupby("film_id")
                   .agg(total_revenue=("amount", "sum"))
                   .reset_index()
                   .merge(film_df[["film_id", "title"]], on="film_id")      # EKLENDİ
                   .sort_values(["total_revenue", "title"], ascending=[True, True])  # DEĞİŞTİ
                   .head(1)["film_id"].iloc[0])

merged = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id"))

result = (merged[merged["film_id"] == bottom_film_id]
          .groupby(["store_id", "title"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index())
print(result)

   store_id             title  total_revenue
0         1  OKLAHOMA JUMANJI           3.96
1         2  OKLAHOMA JUMANJI           1.98


51-) Müşterilerin kiraladıkları filmlerin toplam kiralama süresi ne kadardır?

In [182]:
rental_df["rental_date"] = pd.to_datetime(rental_df["rental_date"])
rental_df["return_date"] = pd.to_datetime(rental_df["return_date"])
rental_df["duration"] = (rental_df["return_date"] - rental_df["rental_date"]) / pd.Timedelta(days=1)

result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .groupby(["first_name", "last_name"])
          .agg(total_rental_days=("duration", "sum"))
          .reset_index())
print(result)

    first_name last_name  total_rental_days
0        AARON     SELBY         120.398611
1         ADAM     GOOCH         116.220833
2       ADRIAN     CLARY         101.916667
3        AGNES    BISHOP         111.152778
4         ALAN      KAHN         143.023611
..         ...       ...                ...
594     WILLIE   MARKHAM         125.833333
595      WILMA  RICHARDS         117.254861
596    YOLANDA    WEAVER         139.776389
597     YVONNE   WATKINS          97.145139
598    ZACHARY      HITE         169.749306

[599 rows x 3 columns]


52-) En çok kiralanan türdeki filmler hangileridir?

In [183]:
top_genre = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
             .merge(film_df.drop(columns=["last_update"]), on="film_id")
             .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
             .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
             .rename(columns={"name": "genre"})
             .groupby("genre")
             .size()
             .reset_index(name="rental_count")
             .sort_values("rental_count", ascending=False)
             .head(1)["genre"].iloc[0])

merged = (film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .rename(columns={"name": "genre"}))

result = (merged[merged["genre"] == top_genre]
          .groupby(["title", "genre"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count","title"], ascending=[False,False]))
print(result.reset_index(drop=True))

                  title   genre  rental_count
0     TALENTED HOMICIDE  Sports            29
1   GLEAMING JAWBREAKER  Sports            29
2        SATURDAY LAMBS  Sports            28
3        ROSES TREASURE  Sports            28
4   CHANCE RESURRECTION  Sports            27
..                  ...     ...           ...
68     LESSON CLEOPATRA  Sports             7
69      DIVORCE SHINING  Sports             7
70   GRACELAND DYNAMITE  Sports             6
71        BUBBLE GROSSE  Sports             6
72   MUSSOLINI SPOILERS  Sports             5

[73 rows x 3 columns]


Burada da çıktıların aynı olması için 2. bir sort parametresi ekledim

53-) En az kiralanan türdeki filmler hangileridir?

In [184]:
bottom_genre = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
                 .merge(film_df.drop(columns=["last_update"]), on="film_id")
                 .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
                 .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
                 .rename(columns={"name": "genre"})
                 .groupby("genre")
                 .size()
                 .reset_index(name="rental_count")
                 .sort_values("rental_count", ascending=True)
                 .head(1)["genre"].iloc[0])

merged = (film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .rename(columns={"name": "genre"}))

result = (merged[merged["genre"] == bottom_genre]
          .groupby(["title", "genre"])
          .size()
          .reset_index(name="rental_count")
          .sort_values("rental_count", ascending=True))
print(result.reset_index(drop=True))

                     title  genre  rental_count
0               YOUTH KICK  Music             6
1          TERMINATOR CLUB  Music             6
2               ELF MURDER  Music             7
3               CLUE GRAIL  Music             7
4             HEAVENLY GUN  Music             7
5           RUNNER MADIGAN  Music             8
6         GROSSE WONDERFUL  Music             8
7            REBEL AIRPORT  Music             8
8              LEGEND JEDI  Music             9
9          BIRDCAGE CASPER  Music             9
10               WON DARES  Music             9
11          IMPACT ALADDIN  Music             9
12              REDS POCUS  Music             9
13     JAWBREAKER BROOKLYN  Music            10
14           OLEANDER CLUE  Music            10
15     AMELIE HELLFIGHTERS  Music            10
16             SONG HEDWIG  Music            11
17         BIRCH ANTITRUST  Music            11
18           MINORITY KISS  Music            12
19         CHAMBER ITALIAN  Music       

54-) Her müşteri için toplam ödeme miktarını bulun

In [185]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(total_payment=("amount", "sum"))
          .reset_index())
print(result)

    first_name last_name  total_payment
0        AARON     SELBY         110.76
1         ADAM     GOOCH         101.78
2       ADRIAN     CLARY          74.81
3        AGNES    BISHOP          98.77
4         ALAN      KAHN         124.74
..         ...       ...            ...
594     WILLIE   MARKHAM         101.75
595      WILMA  RICHARDS          91.80
596    YOLANDA    WEAVER         110.73
597     YVONNE   WATKINS          92.79
598    ZACHARY      HITE         146.69

[599 rows x 3 columns]


55-)Hangi filmler en uzun süre kiralanmış

In [186]:
merged = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id"))
merged["duration"] = (merged["return_date"] - merged["rental_date"]) / pd.Timedelta(days=1)

result = (merged.groupby("title")
          .agg(max_rental_days=("duration", "max"))
          .reset_index()
          .sort_values(["max_rental_days", "title"], ascending=[False, True]))
print(result.reset_index(drop=True))

                  title  max_rental_days
0       HIGHBALL POTTER         9.249306
1    HOLOCAUST HIGHBALL         9.249306
2     CONEHEADS SMOOCHY         9.248611
3            MASK PEACH         9.248611
4            PANIC CLUB         9.248611
..                  ...              ...
953          SQUAD FISH         5.908333
954   CONSPIRACY SPIRIT         5.809722
955         GLORY TRACY         5.120833
956      TRAFFIC HOBBIT         4.977778
957   DUFFEL APOCALYPSE         4.137500

[958 rows x 2 columns]


56-)Hangi filmler en kısa süre kiralanmış

In [187]:
merged = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id"))
merged["duration"] = (merged["return_date"] - merged["rental_date"]) / pd.Timedelta(days=1)

result = (merged.groupby("title")
          .agg(min_rental_days=("duration", "min"))
          .reset_index()
          .sort_values("min_rental_days", ascending=True))
print(result.reset_index(drop=True))

                   title  min_rental_days
0        DISCIPLE MOTHER         0.750000
1      CURTAIN VIDEOTAPE         0.750000
2           ROAD ROXANNE         0.750694
3         STRAIGHT HOURS         0.751389
4    RUGRATS SHAKESPEARE         0.751389
..                   ...              ...
953          MIXED DOORS         3.240972
954           EARLY HOME         3.807639
955      PANKY SUBMARINE         3.827083
956           WAGON JAWS         3.859028
957         ZHIVAGO CORE         3.875000

[958 rows x 2 columns]


57-)Müşterilerin kiraladığı filmler için ortalama ödeme mıktarını bulun

In [188]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(avg_payment=("amount", "mean"))
          .reset_index())
print(result)

    first_name last_name  avg_payment
0        AARON     SELBY     4.615000
1         ADAM     GOOCH     4.626364
2       ADRIAN     CLARY     3.937368
3        AGNES    BISHOP     4.294348
4         ALAN      KAHN     4.797692
..         ...       ...          ...
594     WILLIE   MARKHAM     4.070000
595      WILMA  RICHARDS     4.590000
596    YOLANDA    WEAVER     4.101111
597     YVONNE   WATKINS     4.418571
598    ZACHARY      HITE     4.731935

[599 rows x 3 columns]


58-) En çok kiralanan filmlerin ortalama kiralama süresi nedir?

In [189]:
merged = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id"))
merged["duration"] = (merged["return_date"] - merged["rental_date"]) / pd.Timedelta(days=1)

result = (merged.groupby("title")
          .agg(avg_rental_days=("duration", "mean"), rental_count=("rental_id", "count"))
          .reset_index()
          .sort_values(["rental_count", "title"], ascending=[False, True])
          .head(10)
          .drop(columns=["rental_count"]))
print(result.reset_index(drop=True))

                 title  avg_rental_days
0   BUCKET BROTHERHOOD         4.948448
1     ROCKETEER MOTHER         5.386069
2       FORWARD TEMPLE         5.595247
3       GRIT CLOCKWORK         5.340864
4       JUGGLER HARDLY         5.509901
5  RIDGEMONT SUBMARINE         6.010909
6        SCALAWAG DUCK         4.069640
7        APACHE DIVINE         4.269064
8    GOODFELLAS SALUTE         5.219848
9         HOBBIT ALIEN         5.716711


59-)En az kiralanan filmlerin ortalama kira süresi nedir?

In [190]:
merged = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id"))
merged["duration"] = (merged["return_date"] - merged["rental_date"]) / pd.Timedelta(days=1)

result = (merged.groupby("title")
          .agg(avg_rental_days=("duration", "mean"), rental_count=("rental_id", "count"))
          .reset_index()
          .sort_values(["rental_count", "title"], ascending=[True, True])
          .head(10)
          .drop(columns=["rental_count"]))
print(result.reset_index(drop=True))

               title  avg_rental_days
0     HARDLY ROBBERS         6.948437
1        MIXED DOORS         5.953299
2        TRAIN BUNCH         3.349306
3   BRAVEHEART HUMAN         4.019861
4        BUNCH MINDS         5.524444
5  CONSPIRACY SPIRIT         4.010000
6       FEVER EMPIRE         5.755694
7  FREEDOM CLEOPATRA         2.959722
8    FULL FLATLINERS         3.767361
9        GLORY TRACY         3.678472


60-) En çok kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir?

In [191]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(avg_payment=("amount", "mean"), total_spent=("amount", "sum"))
          .reset_index()
          .sort_values("total_spent", ascending=False)
          .head(10)
          .drop(columns=["total_spent"]))
print(result.reset_index(drop=True))

  first_name last_name  avg_payment
0       KARL      SEAL     4.923333
1    ELEANOR      HUNT     4.707391
2      CLARA      SHAW     4.656667
3     RHONDA   KENNEDY     4.990000
4     MARION    SNYDER     4.990000
5      TOMMY   COLLAZO     4.911053
6     WESLEY      BULL     4.440000
7        TIM      CARY     4.502821
8     MARCIA      DEAN     4.180476
9        ANA   BRADLEY     5.137059


61-) En az kazanç sağlayan müşterilerin ortalama ödeme miktarı nedir

In [192]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(avg_payment=("amount", "mean"), total_spent=("amount", "sum"))
          .reset_index()
          .sort_values(["total_spent", "last_name", "first_name"], ascending=[True, True, True])
          .head(10)
          .drop(columns=["total_spent"])
          .reset_index(drop=True))
print(result)

  first_name last_name  avg_payment
0   CAROLINE    BOWMAN     3.390000
1      LEONA    OBRIEN     3.632857
2      BRIAN     WYMAN     4.406667
3     JOHNNY    TURPIN     3.042632
4      ANNIE   RUSSELL     3.267778
5  KATHERINE    RIVERA     4.204286
6    TIFFANY    JORDAN     4.275714
7      ANITA   MORALES     4.190000
8     MATTIE   HOFFMAN     2.944545
9       KIRK   STCLAIR     3.411053


62-)Mağazalardaki toplam kiralama süresini bulunuz

In [193]:
merged = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update", "address_id", "email", "username", "password", "active"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id"))
merged["duration"] = (pd.to_datetime(merged["return_date"]) - pd.to_datetime(merged["rental_date"])) / pd.Timedelta(days=1)

result = (merged.groupby("store_id")
          .agg(total_rental_days=("duration", "sum"))
          .reset_index()
          .sort_values("store_id")
          .reset_index(drop=True))
print(result)

   store_id  total_rental_days
0         1       39878.755556
1         2       39828.009028


63-)En uzun süre kiralanan film hangi mağazada kiralanmış

In [194]:
merged = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update", "address_id", "email", "username", "password", "active"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id"))
merged["duration"] = ((pd.to_datetime(merged["return_date"]) - pd.to_datetime(merged["rental_date"]))
                       / pd.Timedelta(days=1)).round(6)
result = (merged.groupby(["store_id", "title"])
          .agg(max_rental_days=("duration", "max"))
          .reset_index()
          .sort_values(["max_rental_days", "title"], ascending=[False, True])
          .head(1)
          .reset_index(drop=True))
print(result)

   store_id            title  max_rental_days
0         2  HIGHBALL POTTER         9.249306


Bu soruda SQL ve Pandas çıktıları farklıydı. Kiralama gün sayısı aynı ancak sort parametrelerini aynı vermiş olmama rağmen sonuçlar farklı çıkmıştı ancak daha sonra round ile sorun çözüldü.

64-) En kısa süre kiralanan film hangi mağazada kiralanmış

In [195]:
merged = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update", "address_id", "email", "username", "password", "active"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id"))
merged["duration"] = (pd.to_datetime(merged["return_date"]) - 
                      pd.to_datetime(merged["rental_date"])) / pd.Timedelta(days=1)

result = (merged.groupby(["store_id", "title"])
          .agg(min_rental_days=("duration", "min"))
          .reset_index()
          .sort_values(["min_rental_days", "title"], ascending=[True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

   store_id              title  min_rental_days
0         2  CURTAIN VIDEOTAPE             0.75


65-) Her filmin ortalama kiralama süresi nedir?

In [196]:
merged = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id"))
merged["duration"] = (pd.to_datetime(merged["return_date"]) - 
                      pd.to_datetime(merged["rental_date"])) / pd.Timedelta(days=1)

result = (merged.groupby("title")
          .agg(avg_rental_days=("duration", "mean"))
          .reset_index()
          .sort_values("title")
          .reset_index(drop=True))
print(result)

                 title  avg_rental_days
0     ACADEMY DINOSAUR         4.997128
1       ACE GOLDFINGER         5.642708
2     ADAPTATION HOLES         3.465625
3     AFFAIR PREJUDICE         4.758333
4          AFRICAN EGG         7.106629
..                 ...              ...
953     YOUNG LANGUAGE         4.624405
954         YOUTH KICK         5.562963
955       ZHIVAGO CORE         5.950087
956  ZOOLANDER FICTION         5.542647
957          ZORRO ARK         4.539180

[958 rows x 2 columns]


66-) Hangi filmler en çok kiralanan kategoride

In [197]:
top_genre = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
             .merge(film_df.drop(columns=["last_update"]), on="film_id")
             .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
             .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
             .rename(columns={"name": "genre"})
             .groupby("genre")
             .size()
             .reset_index(name="rental_count")
             .sort_values(["rental_count", "genre"], ascending=[False, True])
             .head(1)["genre"].iloc[0])

merged = (film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .rename(columns={"name": "genre"}))

result = (merged[merged["genre"] == top_genre]
          .groupby(["title", "genre"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "title"], ascending=[False, True])
          .reset_index(drop=True))
print(result)

                  title   genre  rental_count
0   GLEAMING JAWBREAKER  Sports            29
1     TALENTED HOMICIDE  Sports            29
2        ROSES TREASURE  Sports            28
3        SATURDAY LAMBS  Sports            28
4   CHANCE RESURRECTION  Sports            27
..                  ...     ...           ...
68          SENSE GREEK  Sports             7
69           SQUAD FISH  Sports             7
70        BUBBLE GROSSE  Sports             6
71   GRACELAND DYNAMITE  Sports             6
72   MUSSOLINI SPOILERS  Sports             5

[73 rows x 3 columns]


67-)Hangi filmler en az kiralanan kategoride

In [198]:
bottom_genre = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
                 .merge(film_df.drop(columns=["last_update"]), on="film_id")
                 .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
                 .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
                 .rename(columns={"name": "genre"})
                 .groupby("genre")
                 .size()
                 .reset_index(name="rental_count")
                 .sort_values(["rental_count", "genre"], ascending=[True, True])
                 .head(1)["genre"].iloc[0])

merged = (film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .rename(columns={"name": "genre"}))

result = (merged[merged["genre"] == bottom_genre]
          .groupby(["title", "genre"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "title"], ascending=[True, True])
          .reset_index(drop=True))
print(result)

                     title  genre  rental_count
0          TERMINATOR CLUB  Music             6
1               YOUTH KICK  Music             6
2               CLUE GRAIL  Music             7
3               ELF MURDER  Music             7
4             HEAVENLY GUN  Music             7
5         GROSSE WONDERFUL  Music             8
6            REBEL AIRPORT  Music             8
7           RUNNER MADIGAN  Music             8
8          BIRDCAGE CASPER  Music             9
9           IMPACT ALADDIN  Music             9
10             LEGEND JEDI  Music             9
11              REDS POCUS  Music             9
12               WON DARES  Music             9
13     AMELIE HELLFIGHTERS  Music            10
14     JAWBREAKER BROOKLYN  Music            10
15           OLEANDER CLUE  Music            10
16         BIRCH ANTITRUST  Music            11
17             SONG HEDWIG  Music            11
18           MINORITY KISS  Music            12
19         CHAMBER ITALIAN  Music       

68-) Hangi mağazalarda en çok film kiralanmış

In [199]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .groupby("store_id")
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "store_id"], ascending=[False, True])
          .reset_index(drop=True))
print(result)

   store_id  rental_count
0         1          8040
1         2          8004


69-) Hangi mağazalarda en az film kiralanmış

In [200]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .groupby("store_id")
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "store_id"], ascending=[True, True])
          .reset_index(drop=True))
print(result)

   store_id  rental_count
0         2          8004
1         1          8040


70-) Hangi aktörler en çok filmde oynamış

In [201]:
result = (actor_df.drop(columns=["last_update"])
          .merge(film_actor_df.drop(columns=["last_update"]), on="actor_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="film_count")
          .sort_values(["film_count", "last_name", "first_name"], ascending=[False, True, True])
          .reset_index(drop=True))
print(result)

    first_name  last_name  film_count
0        SUSAN      DAVIS          54
1         GINA  DEGENERES          42
2       WALTER       TORN          41
3         MARY     KEITEL          40
4      MATTHEW     CARREY          39
..         ...        ...         ...
194      SISSY   SOBIESKI          18
195      JULIA  ZELLWEGER          16
196       JUDY       DEAN          15
197      JULIA    FAWCETT          15
198      EMILY        DEE          14

[199 rows x 3 columns]


71-) Hangi aktörler en az filmde rol almış

In [202]:
result = (actor_df.drop(columns=["last_update"])
          .merge(film_actor_df.drop(columns=["last_update"]), on="actor_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="film_count")
          .sort_values(["film_count", "last_name", "first_name"], ascending=[True, True, True])
          .reset_index(drop=True))
print(result)

    first_name  last_name  film_count
0        EMILY        DEE          14
1         JUDY       DEAN          15
2        JULIA    FAWCETT          15
3        JULIA  ZELLWEGER          16
4         ADAM      GRANT          18
..         ...        ...         ...
194    MATTHEW     CARREY          39
195       MARY     KEITEL          40
196     WALTER       TORN          41
197       GINA  DEGENERES          42
198      SUSAN      DAVIS          54

[199 rows x 3 columns]


72-) Hangi aktörlerin oynadığı filmler en çok kiralanmış

In [203]:
result = (actor_df.drop(columns=["last_update"])
          .merge(film_actor_df.drop(columns=["last_update"]), on="actor_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "last_name", "first_name"], ascending=[False, True, True])
          .reset_index(drop=True))
print(result)

    first_name    last_name  rental_count
0        SUSAN        DAVIS           825
1         GINA    DEGENERES           753
2      MATTHEW       CARREY           678
3         MARY       KEITEL           674
4       ANGELA  WITHERSPOON           654
..         ...          ...           ...
194       JUDY         DEAN           255
195      JULIA      FAWCETT           255
196      SISSY     SOBIESKI           235
197      JULIA    ZELLWEGER           221
198      EMILY          DEE           216

[199 rows x 3 columns]


73-) Hangi aktörlerin oynadığı filmler en az kiralanmış

In [204]:
result = (actor_df.drop(columns=["last_update"])
          .merge(film_actor_df.drop(columns=["last_update"]), on="actor_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "last_name", "first_name"], ascending=[True, True, True])
          .reset_index(drop=True))
print(result)

    first_name    last_name  rental_count
0        EMILY          DEE           216
1        JULIA    ZELLWEGER           221
2        SISSY     SOBIESKI           235
3         JUDY         DEAN           255
4        JULIA      FAWCETT           255
..         ...          ...           ...
194     ANGELA  WITHERSPOON           654
195       MARY       KEITEL           674
196    MATTHEW       CARREY           678
197       GINA    DEGENERES           753
198      SUSAN        DAVIS           825

[199 rows x 3 columns]


74-)Hangi kategorilerde en fazla aktör oynamış

In [205]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(film_actor_df.drop(columns=["last_update"]), on="film_id")
          .rename(columns={"name": "genre"})
          .groupby("genre")
          .agg(actor_count=("actor_id", "nunique"))
          .reset_index()
          .sort_values(["actor_count", "genre"], ascending=[False, True])
          .reset_index(drop=True))
print(result)

          genre  actor_count
0        Sports          182
1       Foreign          175
2           New          169
3   Documentary          168
4        Sci-Fi          167
5        Action          166
6     Animation          166
7        Travel          166
8        Family          164
9      Children          163
10     Classics          162
11        Drama          162
12       Horror          156
13        Games          150
14       Comedy          147
15        Music          144


75-)Hangi kategorilerde en az aktör oynamış

In [206]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(film_actor_df.drop(columns=["last_update"]), on="film_id")
          .rename(columns={"name": "genre"})
          .groupby("genre")
          .agg(actor_count=("actor_id", "nunique"))
          .reset_index()
          .sort_values(["actor_count", "genre"], ascending=[True, True])
          .reset_index(drop=True))
print(result)

          genre  actor_count
0         Music          144
1        Comedy          147
2         Games          150
3        Horror          156
4      Classics          162
5         Drama          162
6      Children          163
7        Family          164
8        Action          166
9     Animation          166
10       Travel          166
11       Sci-Fi          167
12  Documentary          168
13          New          169
14      Foreign          175
15       Sports          182


76-) Hangi kategorilerde oynayan filmler en çok kiralanmış

In [207]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .rename(columns={"name": "genre"})
          .groupby("genre")
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "genre"], ascending=[False, True])
          .reset_index(drop=True))
print(result)

          genre  rental_count
0        Sports          1179
1     Animation          1166
2        Action          1112
3        Sci-Fi          1101
4        Family          1096
5         Drama          1060
6   Documentary          1050
7       Foreign          1033
8         Games           969
9      Children           945
10       Comedy           941
11          New           940
12     Classics           939
13       Horror           846
14       Travel           837
15        Music           830


77-) Hangi kategorilerde oynayan filmler en az kiralanmış

In [208]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .rename(columns={"name": "genre"})
          .groupby("genre")
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "genre"], ascending=[True, True])
          .reset_index(drop=True))
print(result)

          genre  rental_count
0         Music           830
1        Travel           837
2        Horror           846
3      Classics           939
4           New           940
5        Comedy           941
6      Children           945
7         Games           969
8       Foreign          1033
9   Documentary          1050
10        Drama          1060
11       Family          1096
12       Sci-Fi          1101
13       Action          1112
14    Animation          1166
15       Sports          1179


78-)En fazla kazanç sağlayan kategoriler hangileridir

In [209]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .rename(columns={"name": "genre"})
          .groupby("genre")
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "genre"], ascending=[False, True])
          .reset_index(drop=True))
print(result)

          genre  total_revenue
0        Sports        5314.21
1        Sci-Fi        4756.98
2     Animation        4656.30
3         Drama        4587.39
4        Comedy        4383.58
5        Action        4375.85
6           New        4351.62
7         Games        4281.33
8       Foreign        4270.67
9        Family        4226.07
10  Documentary        4217.52
11       Horror        3722.54
12     Children        3655.55
13     Classics        3639.59
14       Travel        3549.64
15        Music        3417.72


79-) En az kazanç sağlayan kategoriler hangileri

In [210]:
result = (category_df.merge(film_category_df.drop(columns=["last_update"]), on="category_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .rename(columns={"name": "genre"})
          .groupby("genre")
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "genre"], ascending=[True, True])
          .reset_index(drop=True))
print(result)

          genre  total_revenue
0         Music        3417.72
1        Travel        3549.64
2      Classics        3639.59
3      Children        3655.55
4        Horror        3722.54
5   Documentary        4217.52
6        Family        4226.07
7       Foreign        4270.67
8         Games        4281.33
9           New        4351.62
10       Action        4375.85
11       Comedy        4383.58
12        Drama        4587.39
13    Animation        4656.30
14       Sci-Fi        4756.98
15       Sports        5314.21


80-) En çok film kiralayan müşterilerin ortalama ödeme miktarı

In [211]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(avg_payment=("amount", "mean"), rental_count=("rental_id", "count"))
          .reset_index()
          .sort_values(["rental_count", "last_name", "first_name"], ascending=[False, True, True])
          .head(10)
          .drop(columns=["rental_count"])
          .reset_index(drop=True))
print(result)

  first_name last_name  avg_payment
0    ELEANOR      HUNT     4.707391
1       KARL      SEAL     4.923333
2     MARCIA      DEAN     4.180476
3      CLARA      SHAW     4.656667
4      TAMMY   SANDERS     3.794878
5     WESLEY      BULL     4.440000
6        SUE    PETERS     3.865000
7        TIM      CARY     4.502821
8     RHONDA   KENNEDY     4.990000
9     MARION    SNYDER     4.990000


81-) En az film kiralayan müşterilerin ortalama ödeme miktarı

In [212]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(avg_payment=("amount", "mean"), rental_count=("rental_id", "count"))
          .reset_index()
          .sort_values(["rental_count", "last_name", "first_name"], ascending=[True, True, True])
          .head(10)
          .drop(columns=["rental_count"])
          .reset_index(drop=True))
print(result)

  first_name last_name  avg_payment
0      BRIAN     WYMAN     4.406667
1    TIFFANY    JORDAN     4.275714
2      LEONA    OBRIEN     3.632857
3  KATHERINE    RIVERA     4.204286
4   CAROLINE    BOWMAN     3.390000
5      ANITA   MORALES     4.190000
6      JOANN   GARDNER     4.177500
7     JEROME    KENYON     4.615000
8     LESTER     KRAUS     4.115000
9    ANTONIO      MEEK     4.927500


82-) Hangi mağazada en hangi kategoriler en çok kiralanmış

In [213]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .rename(columns={"name": "genre"})
          .groupby(["store_id", "genre"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "store_id", "genre"], ascending=[False, True, True])
          .reset_index(drop=True))
print(result)

    store_id        genre  rental_count
0          2       Sports           614
1          2    Animation           584
2          1    Animation           582
3          1       Family           565
4          1       Sports           565
5          1       Action           562
6          1       Sci-Fi           553
7          2       Action           550
8          2       Sci-Fi           548
9          2      Foreign           543
10         2        Drama           532
11         2       Family           531
12         1        Drama           528
13         1  Documentary           525
14         2  Documentary           525
15         1        Games           494
16         1      Foreign           490
17         1          New           489
18         1     Children           483
19         1     Classics           475
20         1       Comedy           475
21         2        Games           475
22         2       Comedy           466
23         2     Classics           464


83-) Hangi mağazada hangi kategoriler en az kiralanmış

In [214]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .rename(columns={"name": "genre"})
          .groupby(["store_id", "genre"])
          .size()
          .reset_index(name="rental_count")
          .sort_values(["rental_count", "store_id", "genre"], ascending=[True, True, True])
          .reset_index(drop=True))
print(result)

    store_id        genre  rental_count
0          2        Music           398
1          1       Travel           399
2          1       Horror           423
3          2       Horror           423
4          1        Music           432
5          2       Travel           438
6          2          New           451
7          2     Children           462
8          2     Classics           464
9          2       Comedy           466
10         1     Classics           475
11         1       Comedy           475
12         2        Games           475
13         1     Children           483
14         1          New           489
15         1      Foreign           490
16         1        Games           494
17         1  Documentary           525
18         2  Documentary           525
19         1        Drama           528
20         2       Family           531
21         2        Drama           532
22         2      Foreign           543
23         2       Sci-Fi           548


84-) En fazla sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?

In [215]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .groupby("store_id")
          .size()
          .reset_index(name="total_rentals")
          .sort_values(["total_rentals", "store_id"], ascending=[False, True])
          .head(1)
          .reset_index(drop=True))
print(result)

   store_id  total_rentals
0         1           8040


85-)En az sayıda film kiralayan mağaza hangisidir ve toplam kiralama sayısı nedir?

In [216]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .groupby("store_id")
          .size()
          .reset_index(name="total_rentals")
          .sort_values(["total_rentals", "store_id"], ascending=[True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

   store_id  total_rentals
0         2           8004


86-)En fazla sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir

In [217]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="total_rentals")
          .sort_values(["total_rentals", "last_name", "first_name"], ascending=[False, True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

  first_name last_name  total_rentals
0    ELEANOR      HUNT             46


87-) En az sayıda kiralama yapan müşteri hangisidir ve toplam kiralama sayısı nedir?

In [218]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .groupby(["first_name", "last_name"])
          .size()
          .reset_index(name="total_rentals")
          .sort_values(["total_rentals", "last_name", "first_name"], ascending=[True, True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

  first_name last_name  total_rentals
0      BRIAN     WYMAN             12


88-) En fazla sayıda kiralanan film hangisidir ve toplam kiralama sayısı nedir

In [219]:
result = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .groupby("title")
          .size()
          .reset_index(name="total_rentals")
          .sort_values(["total_rentals", "title"], ascending=[False, True])
          .head(1)
          .reset_index(drop=True))
print(result)

                title  total_rentals
0  BUCKET BROTHERHOOD             34

89-) En az sayıda kiiralanan film hangisidir ve toplam kiralama sayısı nedir

In [220]:
result = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .groupby("title")
          .size()
          .reset_index(name="total_rentals")
          .sort_values(["total_rentals", "title"], ascending=[True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

            title  total_rentals
0  HARDLY ROBBERS              4


90-)En fazla kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir

In [221]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "last_name", "first_name"], ascending=[False, True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

  first_name last_name  total_revenue
0       KARL      SEAL         221.55


91-)En az kazanç sağlayan müşteri hangisidir ve toplam kazancı nedir

In [222]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby(["first_name", "last_name"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "last_name", "first_name"], ascending=[True, True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

  first_name last_name  total_revenue
0   CAROLINE    BOWMAN          50.85


92-)Hangi film hangi kategoride en çok kazanç sağlamıştır

In [223]:
agg = (film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
       .merge(category_df.drop(columns=["last_update"]), on="category_id")
       .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
       .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
       .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
       .rename(columns={"name": "genre"})
       .groupby(["genre", "title"])
       .agg(total_revenue=("amount", "sum"))
       .reset_index())

result = (agg.sort_values(["genre", "total_revenue", "title"], ascending=[True, False, True])
          .groupby("genre")
          .head(1)
          .reset_index(drop=True))
print(result)

          genre                title  total_revenue
0        Action     FOOL MOCKINGBIRD         175.77
1     Animation         DOGMA FAMILY         178.70
2      Children  BACKLASH UNDEFEATED         158.81
3      Classics          STEEL SANTA         141.77
4        Comedy            ZORRO ARK         214.69
5   Documentary            WIFE TURN         223.69
6         Drama         TORQUE BOUND         198.72
7        Family     RANGE MOONWALKER         179.73
8       Foreign       INNOCENT USUAL         191.74
9         Games       MASSACRE USUAL         179.70
10       Horror           LOLA AGENT         159.76
11        Music     TELEGRAPH VOYAGE         231.73
12          New          MAIDEN HOME         163.76
13       Sci-Fi    GOODFELLAS SALUTE         209.69
14       Sports       SATURDAY LAMBS         204.72
15       Travel   BUCKET BROTHERHOOD         180.66


93-)Hangi film hangi kategoride en az kazanç sağlamıştır

In [224]:
agg = (film_df.merge(film_category_df.drop(columns=["last_update"]), on="film_id")
       .merge(category_df.drop(columns=["last_update"]), on="category_id")
       .merge(inventory_df.drop(columns=["last_update"]), on="film_id")
       .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
       .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
       .rename(columns={"name": "genre"})
       .groupby(["genre", "title"])
       .agg(total_revenue=("amount", "sum"))
       .reset_index())

result = (agg.sort_values(["genre", "total_revenue", "title"], ascending=[True, True, True])
          .groupby("genre")
          .head(1)
          .reset_index(drop=True))
print(result)

          genre                 title  total_revenue
0        Action     MONTEZUMA COMMAND          11.91
1     Animation      GHOSTBUSTERS ELF          11.91
2      Children     COMANCHEROS ENEMY           9.94
3      Classics    CRUELTY UNFORGIVEN           7.93
4        Comedy     FREEDOM CLEOPATRA           5.95
5   Documentary     DUFFEL APOCALYPSE           6.93
6         Drama         HAROLD FRENCH           9.92
7        Family           SOUP WISDOM          12.92
8       Foreign  BALLROOM MOCKINGBIRD          10.92
9         Games           GLORY TRACY          14.95
10       Horror           TEXAS WATCH           5.94
11        Music         REBEL AIRPORT           7.92
12          New      OKLAHOMA JUMANJI           5.94
13       Sci-Fi     STALLION SUNDANCE           8.93
14       Sports       VICTORY ACADEMY          12.91
15       Travel   VALENTINE VANISHING          16.88


94-)En fazla kazanç sağlayan mağaza hangisidir, toplam kazancı nedir 

In [225]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby("store_id")
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "store_id"], ascending=[False, True])
          .head(1)
          .reset_index(drop=True))
print(result)

   store_id  total_revenue
0         2       33881.94


95-)En az kazanç sağlayan mağaza hangisidir ve toplam kazancı nedir?

In [226]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .groupby("store_id")
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "store_id"], ascending=[True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

   store_id  total_revenue
0         1       33524.62


96-)En fazla sayıda farklı film kiralayan müşteri hangisidir

In [227]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .groupby(["first_name", "last_name"])
          .agg(unique_rentals=("inventory_id", "nunique"))
          .reset_index()
          .sort_values(["unique_rentals", "last_name", "first_name"], ascending=[False, True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

  first_name last_name  unique_rentals
0    ELEANOR      HUNT              46


97-) En az sayıda farklı film kiralayan müşteri hangsidir?

In [228]:
result = (customer_df.merge(rental_df.drop(columns=["last_update"]), on="customer_id")
          .groupby(["first_name", "last_name"])
          .agg(unique_rentals=("inventory_id", "nunique"))
          .reset_index()
          .sort_values(["unique_rentals", "last_name", "first_name"], ascending=[True, True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

  first_name last_name  unique_rentals
0      BRIAN     WYMAN              12


98-)En fazla sayıda farklı müşteri tarafından kiralanan film hangisidir?

In [229]:
result = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .groupby("title")
          .agg(unique_customers=("customer_id", "nunique"))
          .reset_index()
          .sort_values(["unique_customers", "title"], ascending=[False, True])
          .head(1)
          .reset_index(drop=True))
print(result)

                title  unique_customers
0  BUCKET BROTHERHOOD                33


99-)En az sayıda farklı müşteri tarafından kiralanan film hangisidir

In [230]:
result = (film_df.merge(inventory_df.drop(columns=["last_update"]), on="film_id")
          .merge(rental_df.drop(columns=["last_update"]), on="inventory_id")
          .groupby("title")
          .agg(unique_customers=("customer_id", "nunique"))
          .reset_index()
          .sort_values(["unique_customers", "title"], ascending=[True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

            title  unique_customers
0  HARDLY ROBBERS                 4


100-)Hangi mağazada hangi kategoride en fazla kazanç sağlanmıştır

In [231]:
result = (store_df.drop(columns=["last_update"])
          .merge(staff_df.drop(columns=["last_update"]), on="store_id")
          .merge(rental_df.drop(columns=["last_update"]), on="staff_id")
          .merge(inventory_df.drop(columns=["last_update", "store_id"]), on="inventory_id")
          .merge(film_df.drop(columns=["last_update"]), on="film_id")
          .merge(film_category_df.drop(columns=["last_update"]), on="film_id")
          .merge(category_df.drop(columns=["last_update"]), on="category_id")
          .merge(payment_df.drop(columns=["last_update", "customer_id", "staff_id"]), on="rental_id")
          .rename(columns={"name": "genre"})
          .groupby(["store_id", "genre"])
          .agg(total_revenue=("amount", "sum"))
          .reset_index()
          .sort_values(["total_revenue", "store_id", "genre"], ascending=[False, True, True])
          .head(1)
          .reset_index(drop=True))
print(result)

   store_id   genre  total_revenue
0         2  Sports        2761.85


## GÖREV SONU